# Tree-Based Models — Demand Classification

**Models:** Random Forest · Extra Trees  
**Target:** `demand_label` (binary: 0 = low demand, 1 = high demand)  
**Tuning:** Optuna (weighted-F1 objective)  
Both models use `class_weight='balanced'` to handle any residual class imbalance.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    accuracy_score, f1_score,
)
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

plt.style.use('dark_background')
SEED = 42

## 1. Load Data

In [ ]:
train_df = pd.read_csv('../data/splits/train.csv', low_memory=False)
test_df  = pd.read_csv('../data/splits/test.csv',  low_memory=False)

TARGET = 'demand_label'

DROP_COLS = [
    'demand_label', 'demand_label_3', 'demand_score',
    'Price_log', 'Price_original',
    'Price_vs_city_median',
]
raw_cols     = [c for c in train_df.columns if c.endswith('_raw')]
amenity_cols = [c for c in train_df.columns if 'Parsed Amenities' in c]
DROP_COLS   += raw_cols + amenity_cols

feature_cols = [c for c in train_df.columns if c not in DROP_COLS]

X_all  = train_df[feature_cols].fillna(0)
y_all  = train_df[TARGET]
X_test = test_df[feature_cols].fillna(0)
y_test = test_df[TARGET]

# Sanitize column names for LightGBM (remove special JSON characters)
import re
X_all.columns = [re.sub(r"[^A-Za-z0-9_]+", "_", c) for c in X_all.columns]
X_test.columns = [re.sub(r"[^A-Za-z0-9_]+", "_", c) for c in X_test.columns]

X_train, X_val, y_train, y_val = train_test_split(
    X_all, y_all, test_size=0.2, random_state=SEED, stratify=y_all
)

print(f'Train : {X_train.shape} | Val : {X_val.shape} | Test : {X_test.shape}')
print(f'Class balance (train): {y_train.value_counts(normalize=True).round(3).to_dict()}')

---
## 2. Random Forest

In [ ]:
def objective_rf(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators',      50, 300),
        'criterion':         trial.suggest_categorical('criterion', ['gini', 'entropy']),
        'max_depth':         trial.suggest_int('max_depth',         3, 50),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf':  trial.suggest_int('min_samples_leaf',  1, 20),
        'max_features':      trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
    }
    model = RandomForestClassifier(**params, random_state=SEED, class_weight='balanced', n_jobs=-1)
    model.fit(X_train, y_train)
    return f1_score(y_val, model.predict(X_val), average='weighted')

study_rf = optuna.create_study(direction='maximize')
study_rf.optimize(objective_rf, n_trials=100, n_jobs=2)
print(f'Best Val F1 (RF) : {study_rf.best_value:.4f}')
print(f'Best Params      : {study_rf.best_params}')

In [ ]:
X_combined = pd.concat([X_train, X_val])
y_combined = pd.concat([y_train, y_val])

rf_model = RandomForestClassifier(
    **study_rf.best_params, random_state=SEED, class_weight='balanced', n_jobs=-1
)
rf_model.fit(X_combined, y_combined)
print('Random Forest trained.')

In [ ]:
y_pred_rf = rf_model.predict(X_test)

print('=== Random Forest ===')
print(f'Accuracy      : {accuracy_score(y_test, y_pred_rf):.4f}')
print(f'F1 (weighted) : {f1_score(y_test, y_pred_rf, average="weighted"):.4f}')
print(classification_report(y_test, y_pred_rf, target_names=['Low Demand','High Demand']))

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred_rf),
    display_labels=['Low Demand','High Demand']
).plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Confusion Matrix — Random Forest', fontsize=14)
plt.tight_layout(); plt.show()

In [ ]:
# Feature importances (top 20)
fi_rf = pd.Series(rf_model.feature_importances_, index=feature_cols).nlargest(20)
fig, ax = plt.subplots(figsize=(9, 6))
fi_rf.sort_values().plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Top-20 Feature Importances — Random Forest')
ax.set_xlabel('Mean Decrease in Impurity')
plt.tight_layout(); plt.show()

In [ ]:
lc_rf = RandomForestClassifier(**study_rf.best_params, random_state=SEED, class_weight='balanced', n_jobs=-1)
ts, tr_s, val_s = learning_curve(
    lc_rf, X_train, y_train, cv=5, scoring='f1_weighted',
    train_sizes=np.linspace(0.1, 1.0, 10), n_jobs=-1,
)
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(ts, tr_s.mean(axis=1), marker='o', label='Train F1')
ax.fill_between(ts, tr_s.mean(1)-tr_s.std(1), tr_s.mean(1)+tr_s.std(1), alpha=0.2)
ax.plot(ts, val_s.mean(axis=1), marker='s', label='Val F1')
ax.fill_between(ts, val_s.mean(1)-val_s.std(1), val_s.mean(1)+val_s.std(1), alpha=0.2)
ax.set(xlabel='Training Size', ylabel='F1 (Weighted)', title='Learning Curve — Random Forest')
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

---
## 3. Extra Trees

In [ ]:
def objective_et(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators',      100, 400),
        'max_depth':         trial.suggest_int('max_depth',         5,   50),
        'min_samples_split': trial.suggest_int('min_samples_split', 2,   20),
        'min_samples_leaf':  trial.suggest_int('min_samples_leaf',  1,   20),
        'max_features':      trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
    }
    model = ExtraTreesClassifier(**params, random_state=SEED, class_weight='balanced', n_jobs=-1)
    model.fit(X_train, y_train)
    return f1_score(y_val, model.predict(X_val), average='weighted')

study_et = optuna.create_study(direction='maximize')
study_et.optimize(objective_et, n_trials=50, n_jobs=2)
print(f'Best Val F1 (ET) : {study_et.best_value:.4f}')
print(f'Best Params      : {study_et.best_params}')

In [ ]:
et_model = ExtraTreesClassifier(
    **study_et.best_params, random_state=SEED, class_weight='balanced', n_jobs=-1
)
et_model.fit(X_combined, y_combined)
print('Extra Trees trained.')

In [ ]:
y_pred_et = et_model.predict(X_test)

print('=== Extra Trees ===')
print(f'Accuracy      : {accuracy_score(y_test, y_pred_et):.4f}')
print(f'F1 (weighted) : {f1_score(y_test, y_pred_et, average="weighted"):.4f}')
print(classification_report(y_test, y_pred_et, target_names=['Low Demand','High Demand']))

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred_et),
    display_labels=['Low Demand','High Demand']
).plot(ax=ax, colorbar=False, cmap='Greens')
ax.set_title('Confusion Matrix — Extra Trees', fontsize=14)
plt.tight_layout(); plt.show()

In [ ]:
lc_et = ExtraTreesClassifier(**study_et.best_params, random_state=SEED, class_weight='balanced', n_jobs=-1)
ts, tr_s, val_s = learning_curve(
    lc_et, X_train, y_train, cv=5, scoring='f1_weighted',
    train_sizes=np.linspace(0.1, 1.0, 10), n_jobs=-1,
)
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(ts, tr_s.mean(axis=1), marker='o', label='Train F1')
ax.fill_between(ts, tr_s.mean(1)-tr_s.std(1), tr_s.mean(1)+tr_s.std(1), alpha=0.2)
ax.plot(ts, val_s.mean(axis=1), marker='s', label='Val F1')
ax.fill_between(ts, val_s.mean(1)-val_s.std(1), val_s.mean(1)+val_s.std(1), alpha=0.2)
ax.set(xlabel='Training Size', ylabel='F1 (Weighted)', title='Learning Curve — Extra Trees')
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

---
## 4. Model Comparison

In [ ]:
results = pd.DataFrame({
    'Model':    ['Random Forest', 'Extra Trees'],
    'Accuracy': [accuracy_score(y_test, y_pred_rf), accuracy_score(y_test, y_pred_et)],
    'F1 (weighted)': [
        f1_score(y_test, y_pred_rf, average='weighted'),
        f1_score(y_test, y_pred_et, average='weighted'),
    ],
}).set_index('Model')

print(results.round(4))

results.plot(kind='bar', figsize=(8, 5), rot=0, colormap='viridis')
plt.title('RF vs Extra Trees — Test Set Performance')
plt.ylabel('Score'); plt.ylim(0.5, 1.0); plt.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()